# 🟡 Solution: SAT Convex Polygon Overlap

**Primitive:** `np.roll` for cyclic edges, `@` for batch projections, `min`/`max` per axis

**Reduction:** overlap is True iff for every axis (edge normal from either polygon), the projection intervals `[min_a, max_a]` and `[min_b, max_b]` overlap — i.e., `max_a >= min_b AND max_b >= min_a` for all axes.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitive: np.roll edge normals + matrix projection

def sat_overlap(poly_a, poly_b):
    def edge_normals(poly):
        edges = np.roll(poly, -1, axis=0) - poly          # (K, 2) cyclic edges
        return np.stack([-edges[:, 1], edges[:, 0]], axis=1)  # (K, 2) perp normals

    axes = np.vstack([edge_normals(poly_a), edge_normals(poly_b)])  # (V+W, 2)

    proj_a = poly_a @ axes.T   # (V, V+W) — dot each vertex with each axis
    proj_b = poly_b @ axes.T   # (W, V+W)

    # A separating axis exists iff max_a < min_b OR max_b < min_a on that axis
    separated = (proj_a.max(0) < proj_b.min(0)) | (proj_b.max(0) < proj_a.min(0))
    return not separated.any()

In [ ]:
# 🔍 Verify solution
square = np.array([[0.0,0.0],[1.0,0.0],[1.0,1.0],[0.0,1.0]])
tri_in  = np.array([[0.5,0.5],[1.5,0.5],[1.0,1.5]])
tri_out = np.array([[2.0,0.0],[3.0,0.0],[2.5,1.0]])
print("overlapping:", sat_overlap(square, tri_in))   # expect True
print("separated:  ", sat_overlap(square, tri_out))  # expect False

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: identical squares ─────────────────────────────────────────────
sq = np.array([[0.0,0.0],[1.0,0.0],[1.0,1.0],[0.0,1.0]])
assert sat_overlap(sq, sq.copy()) == True, "Identical squares → overlap"
print("Test 1 passed: identical squares")

# ── Test 2: non-overlapping squares ───────────────────────────────────────
sq2 = np.array([[5.0,0.0],[6.0,0.0],[6.0,1.0],[5.0,1.0]])
assert sat_overlap(sq, sq2) == False, "Separated on X → no overlap"
sq3 = np.array([[0.0,5.0],[1.0,5.0],[1.0,6.0],[0.0,6.0]])
assert sat_overlap(sq, sq3) == False, "Separated on Y → no overlap"
print("Test 2 passed: non-overlapping squares")

# ── Test 3: half-overlapping squares ─────────────────────────────────────
a = np.array([[0.0,0.0],[2.0,0.0],[2.0,2.0],[0.0,2.0]])
b = np.array([[1.0,0.0],[3.0,0.0],[3.0,2.0],[1.0,2.0]])
assert sat_overlap(a, b) == True, "Half-overlapping → overlap"
print("Test 3 passed: half-overlapping squares")

# ── Test 4: triangle vs square ────────────────────────────────────────────
square4 = np.array([[0.0,0.0],[4.0,0.0],[4.0,4.0],[0.0,4.0]])
tri_in  = np.array([[1.0,1.0],[3.0,1.0],[2.0,3.0]])
tri_out = np.array([[4.01,0.0],[6.0,0.0],[5.0,2.0]])
assert sat_overlap(square4, tri_in) == True,  "Triangle inside square → overlap"
assert sat_overlap(square4, tri_out) == False, "Triangle just outside → no overlap"
print("Test 4 passed: triangle vs square")

# ── Test 5: symmetry + timing on 100 random pairs ─────────────────────────
def rand_convex(rng, n=5, s=3.0):
    angles = np.sort(rng.uniform(0, 2*np.pi, n))
    r = rng.uniform(0.5, s, n)
    cx, cy = rng.uniform(0, 10), rng.uniform(0, 10)
    return np.stack([cx + r*np.cos(angles), cy + r*np.sin(angles)], axis=1)

rng = np.random.default_rng(99)
t0 = time.time()
for _ in range(100):
    pa, pb = rand_convex(rng), rand_convex(rng)
    r1, r2 = sat_overlap(pa, pb), sat_overlap(pb, pa)
    assert r1 == r2, f"Not symmetric: {r1} vs {r2}"
elapsed = time.time() - t0
assert elapsed < 3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: 100 random pairs, symmetric ({elapsed:.3f}s)")

print("\nAll tests passed!")